In [2]:
!pip install -q git+https://github.com/zhouy185/rosa-starter.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import numpy as np
from starter import ZONES, TIME_BLOCKS, COSTS, PROMISE, delivery_times

In [4]:
times = delivery_times("Far West", "Fri/Sat eve", 45)
print(len(times), "orders; average delivery time", round(times.mean(), 1),
"minutes")

210 orders; average delivery time 43.9 minutes


#Part I. Describe: How bad is it, really?

##Part 1a: Percentage of late orders

In [5]:
def late_order_percent(zone, time_block, promise):
  times = []
  if zone == "all":
    wanted_zones = ZONES
  else:
    wanted_zones = [zone]

  if time_block == "all":
    wanted_blocks = TIME_BLOCKS
  else:
    wanted_blocks = [time_block]

  for z in wanted_zones:
    for b in wanted_blocks:
        d_times = delivery_times(z, b, promise)
        times.extend(d_times)

  times = np.array(times)

  late_percent = np.mean(times > promise) * 100
  return late_percent


Testing late_order_percent

In [6]:
print("Far West, Fri/Sat eve:", late_order_percent("Far West", "Fri/Sat eve", 45))

Far West, Fri/Sat eve: 42.38095238095238


In [7]:
print("All zones, Lunch:", late_order_percent("all", "Lunch", 45))

All zones, Lunch: 5.087719298245614


In [8]:
print("Far West, all time blocks:", late_order_percent("Far West", "all", 45))

Far West, all time blocks: 19.36416184971098


In [9]:
print("all zones, all time blocks:", late_order_percent("all", "all", 45))

all zones, all time blocks: 10.696517412935323



## Part 1 b: Ranking pairs in descending order of arrival time

In [27]:
late_rates = []
for z in ZONES:
    for b in TIME_BLOCKS:
      rate = late_order_percent(z, b, 45)
      late_rates.append((rate, z, b))
late_rates.sort(reverse=True)

i=0
while i<len(late_rates):
  print(late_rates[i][1], late_rates[i][2], late_rates[i][0])
  i +=1

Far West Fri/Sat eve 40.0
North Fri/Sat eve 28.000000000000004
Far West Weekday eve 18.823529411764707
Far West Lunch 15.0
Central Fri/Sat eve 12.105263157894736
North Weekday eve 10.8
Central Weekday eve 6.969696969696971
North Lunch 6.315789473684211
Far West Other 6.25
Central Lunch 3.4615384615384617
Central Other 2.631578947368421
North Other 1.25


##Part 1c: Average delivery time

In [30]:
def mean_time(zone, time_block, promise):
  if zone == "all":
    wanted_zones = ZONES
  else:
    wanted_zones = [zone]

  if time_block == "all":
    wanted_blocks = TIME_BLOCKS
  else:
    wanted_blocks = [time_block]

  for z in wanted_zones:
    for b in wanted_blocks:
        d_times = delivery_times(z, b, promise, seed = 1) #gets an array of the delivery time for that particular block and zone

  d_times = np.array(d_times)

  return np.mean(d_times)


testing mean_time

In [31]:
print("Far West, Fri/Sat eve:", mean_time("Far West", "Fri/Sat eve", 45))

Far West, Fri/Sat eve: 42.9195238095238


In [32]:
print("All zones, Lunch:", mean_time("all", "Lunch", 45))

All zones, Lunch: 30.58083333333333


In [33]:
print("Far West, all time blocks:", mean_time("Far West", "all", 45))

Far West, all time blocks: 30.04010416666667


In [34]:
print("all zones, all time blocks:", mean_time("all", "all", 45))

all zones, all time blocks: 30.04010416666667


## Part 1d: Ranking pairs in descending order of mean delivery time

In [37]:
mean_times = []
for z in ZONES:
    for b in TIME_BLOCKS:
      t = mean_time(z, b, 45)
      mean_times.append((t, z, b))
mean_times.sort(reverse=True)
i = 0
while i<len(mean_times):
  print(mean_times[i][1], mean_times[i][2], mean_times[i][0])
  i +=1

Far West Fri/Sat eve 42.9195238095238
North Fri/Sat eve 36.76533333333334
Far West Weekday eve 35.044117647058826
Central Fri/Sat eve 32.13263157894737
North Weekday eve 31.3756
Far West Lunch 30.58083333333333
Far West Other 30.04010416666667
Central Weekday eve 29.030909090909088
North Lunch 28.672631578947364
North Other 27.655624999999997
Central Lunch 26.798076923076923
Central Other 26.03684210526316


## Part 1e


The late rate is more relevant because it directly measures how often Rosa fails to meet her 45-minute promise. Although both measures identify Far West on Fri/Sat evenings as the main problem, average delivery times can hide late orders. Therefore, late rate provides a clearer measure of the delivery-performance problem Rosa needs to address

#Part 2: Exploring the best promise

##Part 2a

In [17]:
def late_order_cost(costs):
    refund = costs["refund"]
    churn = costs["churn_orders"]
    margin = costs["margin"]

    lost_profit = churn * margin
    total_cost = refund + lost_profit

    return total_cost

Testing late_order_cost


In [18]:
print("Cost per late order:", late_order_cost(COSTS))

Cost per late order: 26.2


##Part 2b



The range of promised delivery times should be wide enough to capture the trade-off between demand and late-delivery costs. The range can initially be centred around the current 45-minute promise and tested in 5-minute increments.

In [19]:
def best_promise(zone, time_block, promises, costs):
  results = []
  cost_per_late = late_order_cost(costs)
  for p in promises:
    times = delivery_times(zone, time_block, p, seed = 1)
    late_orders = times > p
    number_late = np.sum(late_orders)
    order_profit = len(times) * costs["margin"]
    late_cost = number_late * cost_per_late
    net_profit = order_profit - late_cost
    results.append((net_profit, p))
  results.sort(reverse=True)
  best_net_profit = results[0][0]
  best_time = results[0][1]

  return best_time, best_net_profit


Testing best_promise



In [20]:
promises = list(range(30, 61, 5))
best_time, best_profit = best_promise(
    "Far West",
    "Fri/Sat eve",
    promises,
    COSTS
)

print("Recommended promise:", best_time, "minutes")
print("Net profit: $", round(best_profit, 2))

Recommended promise: 55 minutes
Net profit: $ 1212.4


In [21]:
promises = list(range(30, 80, 5))
best_time, best_profit = best_promise(
    "Far West",
    "Fri/Sat eve",
    promises,
    COSTS
)

print("Recommended promise:", best_time, "minutes")
print("Net profit: $", round(best_profit, 2))

Recommended promise: 55 minutes
Net profit: $ 1212.4
